<a href="https://colab.research.google.com/github/vituhaa/Healthy-Posture/blob/project/EfficientNetB0_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Работа с EfficientNetB0

EfficientNetB0 (Transfer Learning): This model uses a pre-trained EfficientNetB0 backbone with its weights frozen. A custom classification head—consisting of a Global Average Pooling layer, a Dense layer, and a Dropout layer—was added on top for the specific task of posture classification.

Optimizer: Adam

Loss Function: Binary Crossentropy

Epochs: 15

Data Augmentation: To improve generalization, the training data was augmented
with random horizontal flips, rotations, and zooms.

Both models were evaluated on an unseen test set of 27 images. While they achieved the same overall accuracy, their performance characteristics on a per-class basis were notably different.

Transfer learning tutorial - https://www.tensorflow.org/tutorials/images/transfer_learning

In [15]:
import tensorflow as tf
from matplotlib import pyplot as plt
import tensorflow.keras as keras
import os
import numpy as np

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

model = EfficientNetB0(weights='imagenet')

In [3]:
from google.colab import drive
drive.mount('content/')

Mounted at content/


In [ ]:
categories = ['good', 'bad']

train_dir = '/content/content/MyDrive/датасет для курсовой работы, 3 курс/EfficientNetB0_dataset/train'
test_dir = '/content/content/MyDrive/датасет для курсовой работы, 3 курс/EfficientNetB0_dataset/test'

batch_size = 32
picture_size = 224

train_dataset = keras.utils.image_dataset_from_directory(train_dir, shuffle=True, batch_size=batch_size, image_size=(picture_size, picture_size))
test_dataset = keras.utils.image_dataset_from_directory(test_dir, shuffle=True, batch_size=batch_size, image_size=(picture_size, picture_size))

In [ ]:
print(train_dataset)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
  print(labels)
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    if (labels[i] == 0):
      plt.title("bad")
    else:
      plt.title("good")
    plt.axis("off")

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2)
])

for image, _ in train_dataset.take(1):
  first_image = image[0]
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    aug_img = augmentation(tf.expand_dims(first_image, 0))
    plt.imshow(aug_img[0] / 255)
    plt.axis("off")

In [57]:
# inputs = layers.Input(shape=(picture_size, picture_size, 3))
# base_model = EfficientNetB0(include_top=False, input_tensor=inputs, weights='imagenet')
# base_model.trainable = False
# x = layers.GlobalAveragePooling2D()(base_model.output)
# x = layers.Dropout(0.2)(x)
# outputs = layers.Dense(1, activation="softmax")(x)
# print(outputs)

<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_3859>


In [63]:
from keras import layers

def build_base_model(num_classes):
  inputs = layers.Input(shape=(picture_size, picture_size, 3))
  base_model = EfficientNetB0(include_top=False, input_tensor=inputs, weights='imagenet')

  base_model.trainable = False

  x = layers.GlobalAveragePooling2D()(base_model.output)
  x = layers.Dropout(0.2)(x)
  outputs = layers.Dense(num_classes, activation="sigmoid")(x)
  print(outputs)
  base_model = keras.Model(inputs, outputs)
  optimizer = keras.optimizers.Adam(learning_rate=0.001)
  base_model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
  return base_model

In [59]:
def plot_hist(hist):
    plt.plot(hist.history["accuracy"])
    plt.plot(hist.history["val_accuracy"])
    plt.title("model accuracy")
    plt.ylabel("accuracy")
    plt.xlabel("epoch")
    plt.legend(["train", "validation"], loc="upper left")
    plt.show()

In [ ]:
print(train_dataset)
print(test_dataset)

In [ ]:
efficientnet_model = build_base_model(1)

epochs = 15
history = efficientnet_model.fit(train_dataset, epochs=epochs, validation_data=test_dataset)
plot_hist(history)

In [ ]:
print(efficientnet_model.layers[-20])

In [ ]:
for layer in efficientnet_model.layers[-20]:
  if not isinstance(layer, layers.BatchNormalization):
    layer.trainable = True
optimizer = keras.optimizers.Adam(learning_rate=1e-5)
efficientnet_model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

epochs = 4
history_1 = efficientnet_model.fit(train_dataset, epochs=epochs, validation_data=test_dataset)
plot_hist(history_1)

Custom CNN: A lightweight Convolutional Neural Network built from scratch. The architecture consists of three convolutional blocks with MaxPooling, followed by a dense classifier with a Dropout layer for regularization.